# Decision Tree End-to-End — Baseline Eksperimen

Notebook mandiri ini merekam alur eksperimen ID3, C4.5, CART, CART maksimal dengan cost-complexity pruning, evaluasi holdout, dan ekspor submission berversi. Seluruh implementasi algoritma dan preprocessing disalin ke notebook; tidak ada import dari `src/`.

Jalankan semua sel dari root repository. Data dibaca dari `data/`. CART maksimal memakai tuning 5-fold dan dapat memerlukan waktu lebih lama. Baseline tersimpan: CART maksimal + pruning, `ccp_alpha=0.0001`, threshold `0.62`, macro F1 holdout **0.8371**, OOF macro F1 **0.8320**.

In [ ]:
from pathlib import Path
from itertools import combinations
from dataclasses import dataclass
from copy import copy, deepcopy
import filecmp
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

ROOT = Path.cwd()
OUTPUT = ROOT / "output" / "experiments" / "decision_tree"
if not all((ROOT / "data" / filename).exists() for filename in ("train.csv", "test.csv", "sample_submission.csv")):
    raise FileNotFoundError("Jalankan notebook dari root repository yang memiliki data/train.csv, data/test.csv, dan data/sample_submission.csv")


## Implementasi preprocessing dan pohon dari nol

Termasuk ID3, C4.5, CART, probabilitas daun, serta cost-complexity pruning.

In [ ]:
"""Shared preprocessing and small from-scratch ID3, C4.5, and CART baselines."""

from dataclasses import dataclass
from copy import copy, deepcopy
from itertools import combinations

import numpy as np
import pandas as pd


MISSING = "__MISSING__"
UNKNOWN = "__UNKNOWN__"


class SharedPreprocessor:
    """Median-impute numeric data and retain nominal categories as strings."""

    def fit(self, X):
        self.columns = list(X.columns)
        self.numeric = [c for c in self.columns if pd.api.types.is_numeric_dtype(X[c])]
        self.categorical = [c for c in self.columns if c not in self.numeric]
        self.numeric_fill = {
            c: X[c].median() if X[c].notna().any() else 0.0
            for c in self.numeric
        }
        self.categories = {
            c: set(X[c].dropna().astype(str).unique()) for c in self.categorical
        }
        return self

    def transform(self, X):
        result = X.reindex(columns=self.columns).copy()
        for col in self.numeric:
            result[col] = pd.to_numeric(result[col], errors="coerce").fillna(self.numeric_fill[col])
        for col in self.categorical:
            values = result[col].astype("string")
            values = values.fillna(MISSING)
            result[col] = values.where(values.isin(self.categories[col]) | values.eq(MISSING), UNKNOWN).astype(object)
        return result

    def fit_transform(self, X):
        return self.fit(X).transform(X)


def _entropy(zeros, ones):
    total = zeros + ones
    if total == 0:
        return 0.0
    p = zeros / total
    q = ones / total
    return -(p * np.log2(p) if p else 0.0) - (q * np.log2(q) if q else 0.0)


def _gini(zeros, ones):
    total = zeros + ones
    if total == 0:
        return 0.0
    p = zeros / total
    q = ones / total
    return 1.0 - p * p - q * q


@dataclass
class Node:
    prediction: int
    positive_probability: float
    n_samples: int
    feature: str | None = None
    kind: str | None = None
    threshold: float | None = None
    categories: frozenset | None = None
    branches: dict | None = None


class BaselineTree:
    """Greedy binary classifier using information gain, gain ratio, or Gini."""

    def __init__(self, criterion="gini", max_depth=8, min_samples_leaf=40, min_samples_split=80):
        if criterion not in {"information_gain", "gain_ratio", "gini"}:
            raise ValueError("criterion must be information_gain, gain_ratio, or gini")
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split

    def fit(self, X, y):
        self.columns = list(X.columns)
        self.numeric = [c for c in self.columns if pd.api.types.is_numeric_dtype(X[c])]
        self.values = {c: X[c].to_numpy() for c in self.columns}
        self.y = np.asarray(y, dtype=np.int8)
        if set(np.unique(self.y)) - {0, 1}:
            raise ValueError("BaselineTree expects binary labels 0/1")
        self.root = self._grow(np.arange(len(self.y)), 0)
        return self

    def _impurity(self, zero, one):
        return _gini(zero, one) if self.criterion == "gini" else _entropy(zero, one)

    def _score(self, groups, parent_impurity, parent_size):
        if len(groups) < 2 or any(len(group) < self.min_samples_leaf for group in groups):
            return None
        weighted = sum(len(group) * self._impurity(np.sum(self.y[group] == 0), np.sum(self.y[group] == 1)) for group in groups) / parent_size
        gain = parent_impurity - weighted
        if gain <= 1e-12:
            return None
        if self.criterion != "gain_ratio":
            return gain
        split_info = _entropy(0, 0)
        for group in groups:
            p = len(group) / parent_size
            split_info -= p * np.log2(p)
        return gain / split_info if split_info else None

    def _numeric_candidates(self, indices, col, parent_impurity):
        values = self.values[col][indices].astype(float)
        order = np.argsort(values, kind="stable")
        sorted_values = values[order]
        if len(sorted_values) < 2:
            return
        cuts = np.flatnonzero(sorted_values[:-1] < sorted_values[1:]) + 1
        if len(cuts) > 64:
            cuts = np.unique(cuts[np.linspace(0, len(cuts) - 1, 64).astype(int)])
        for cut in cuts:
            if cut < self.min_samples_leaf or len(indices) - cut < self.min_samples_leaf:
                continue
            threshold = (sorted_values[cut - 1] + sorted_values[cut]) / 2
            left = indices[order[:cut]]
            right = indices[order[cut:]]
            score = self._score((left, right), parent_impurity, len(indices))
            if score is not None:
                yield score, col, "numeric", threshold, None, {"le": left, "gt": right}

    def _categorical_candidates(self, indices, col, parent_impurity):
        values = self.values[col][indices].astype(str)
        categories = np.unique(values)
        if len(categories) < 2:
            return
        if self.criterion != "gini":
            groups = {category: indices[values == category] for category in categories}
            score = self._score(tuple(groups.values()), parent_impurity, len(indices))
            if score is not None:
                yield score, col, "multiway", None, None, groups
            return

        # CART tests unique binary partitions. Above 10 levels, use one-vs-rest
        # candidates to avoid exponential work on high-cardinality features.
        if len(categories) <= 10:
            # Canonical partitions always contain the first level on the left.
            subsets = (frozenset([categories[0], *combo]) for size in range(len(categories)) for combo in combinations(categories[1:], size))
        else:
            subsets = (frozenset([category]) for category in categories)
        seen = set()
        for left_categories in subsets:
            if len(left_categories) == len(categories):
                continue
            key = tuple(sorted(left_categories))
            if key in seen:
                continue
            seen.add(key)
            left_mask = np.isin(values, list(left_categories))
            left, right = indices[left_mask], indices[~left_mask]
            score = self._score((left, right), parent_impurity, len(indices))
            if score is not None:
                yield score, col, "binary_category", None, left_categories, {"left": left, "right": right}

    def _grow(self, indices, depth):
        positives = int(self.y[indices].sum())
        negatives = len(indices) - positives
        majority = int(positives > negatives)
        node = Node(majority, positives / len(indices), len(indices))
        if (self.max_depth is not None and depth >= self.max_depth) or len(indices) < self.min_samples_split or positives == 0 or negatives == 0:
            return node
        parent_impurity = self._impurity(negatives, positives)
        best = None
        for col in self.columns:
            candidates = self._numeric_candidates(indices, col, parent_impurity) if col in self.numeric else self._categorical_candidates(indices, col, parent_impurity)
            for candidate in candidates:
                if best is None or candidate[0] > best[0]:
                    best = candidate
        if best is None:
            return node
        _, node.feature, node.kind, node.threshold, node.categories, groups = best
        node.branches = {key: self._grow(group, depth + 1) for key, group in groups.items()}
        return node

    def _predict_one(self, row):
        node = self.root
        while node.branches is not None:
            value = row[node.feature]
            if node.kind == "numeric":
                key = "le" if float(value) <= node.threshold else "gt"
            elif node.kind == "binary_category":
                key = "left" if str(value) in node.categories else "right"
            else:
                key = str(value)
            child = node.branches.get(key)
            if child is None:
                return node.prediction
            node = child
        return node.prediction

    def predict(self, X):
        return [int(self._predict_one(row)) for row in X.to_dict(orient="records")]

    def _positive_probability_one(self, row):
        node = self.root
        while node.branches is not None:
            value = row[node.feature]
            if node.kind == "numeric":
                key = "le" if float(value) <= node.threshold else "gt"
            elif node.kind == "binary_category":
                key = "left" if str(value) in node.categories else "right"
            else:
                key = str(value)
            child = node.branches.get(key)
            if child is None:
                return node.positive_probability
            node = child
        return node.positive_probability

    def predict_proba(self, X):
        positive = np.asarray([
            self._positive_probability_one(row)
            for row in X.to_dict(orient="records")
        ])
        return np.column_stack((1.0 - positive, positive))

    def _prune_node(self, node, alpha, total_samples):
        leaf_error = node.n_samples * min(node.positive_probability, 1.0 - node.positive_probability) / total_samples
        if node.branches is None:
            return leaf_error, 1

        subtree_error = 0.0
        leaves = 0
        for child in node.branches.values():
            child_error, child_leaves = self._prune_node(child, alpha, total_samples)
            subtree_error += child_error
            leaves += child_leaves

        if leaf_error + alpha <= subtree_error + alpha * leaves:
            node.feature = node.kind = node.threshold = node.categories = None
            node.branches = None
            return leaf_error, 1
        return subtree_error, leaves

    def pruned(self, alpha):
        if not np.isfinite(alpha) or alpha < 0:
            raise ValueError("alpha must be finite and non-negative")
        result = copy(self)
        result.root = deepcopy(self.root)
        self._prune_node(result.root, alpha, len(self.y))
        return result

    def leaf_count(self):
        def count(node):
            return 1 if node.branches is None else sum(count(child) for child in node.branches.values())

        return count(self.root)


## Alur eksperimen

Cell berikut menyalin fungsi eksperimen, tuning CV, evaluasi, serta versioning submission. Preprocessing di-fit per fold; holdout tidak dipakai untuk memilih parameter.

In [ ]:
VARIANTS = {
    "ID3": ("information_gain", "entropy", 0.0),
    "C4.5": ("gain_ratio", "entropy", 0.001),
    "CART": ("gini", "gini", 0.0),
}


def validation_indices(y):
    y = np.asarray(y)
    rng = np.random.default_rng(42)
    valid = np.sort(np.concatenate([
        rng.permutation(np.flatnonzero(y == label))[:int(np.ceil(0.2 * np.sum(y == label)))]
        for label in (0, 1)
    ]))
    train = np.setdiff1d(np.arange(len(y)), valid)
    return train, valid


def threshold_predictions(probabilities, threshold):
    return (np.asarray(probabilities) >= threshold).astype(np.int8).tolist()


def prepare_fold(X_train, X_valid):
    preprocessor = SharedPreprocessor().fit(X_train)
    return preprocessor.transform(X_train), preprocessor.transform(X_valid)


def tune_cart_pruning(X, y, n_splits=5):
    y = np.asarray(y, dtype=np.int8)
    alphas = (0.0, 0.0001, 0.0005, 0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1, 0.25)
    thresholds = np.round(np.arange(0.20, 0.8001, 0.02), 2)
    probabilities = {alpha: np.empty(len(y), dtype=float) for alpha in alphas}
    leaves = {alpha: [] for alpha in alphas}
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for train_idx, valid_idx in splitter.split(X, y):
        X_train, X_valid = prepare_fold(X.iloc[train_idx], X.iloc[valid_idx])
        maximal_tree = BaselineTree(
            criterion="gini", max_depth=None,
            min_samples_split=2, min_samples_leaf=1,
        ).fit(X_train, y[train_idx])
        for alpha in alphas:
            tree = maximal_tree.pruned(alpha)
            probabilities[alpha][valid_idx] = tree.predict_proba(X_valid)[:, 1]
            leaves[alpha].append(tree.leaf_count())

    results = []
    for alpha in alphas:
        for threshold in thresholds:
            prediction = threshold_predictions(probabilities[alpha], threshold)
            results.append({
                "alpha": alpha,
                "threshold": float(threshold),
                "macro_f1": f1_score(y, prediction, labels=[0, 1], average="macro", zero_division=0),
                "mean_leaves": float(np.mean(leaves[alpha])),
            })
    best = max(results, key=lambda result: (result["macro_f1"], result["alpha"], result["threshold"]))
    return best, pd.DataFrame(results)


def score_row(name, y_true, prediction, seconds, detail):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, prediction, labels=[0, 1], zero_division=0
    )
    matrix = confusion_matrix(y_true, prediction, labels=[0, 1])
    return {
        "model": name,
        "validation_macro_f1": f1_score(y_true, prediction, labels=[0, 1], average="macro", zero_division=0),
        "class_0_precision": precision[0],
        "class_0_recall": recall[0],
        "class_0_f1": f1[0],
        "class_1_precision": precision[1],
        "class_1_recall": recall[1],
        "class_1_f1": f1[1],
        "tn": matrix[0, 0], "fp": matrix[0, 1], "fn": matrix[1, 0], "tp": matrix[1, 1],
        "seconds": seconds,
        "configuration": detail,
    }


def build_submission(template, predictions):
    submission = template.copy()
    submission["is_canceled"] = predictions
    return submission


def archive_current_submission(destination):
    destination = Path(destination)
    versions = destination / "versions"
    current = destination / "submission.csv"
    version_files = list(versions.glob("submission_v*.csv"))

    def next_number():
        numbers = [int(path.stem.removeprefix("submission_v")) for path in version_files]
        return max(numbers, default=0) + 1

    if current.exists() and (
        not version_files or not filecmp.cmp(current, max(version_files, key=lambda path: path.name), shallow=False)
    ):
        versions.mkdir(parents=True, exist_ok=True)
        previous = versions / f"submission_v{next_number():03}.csv"
        shutil.copy2(current, previous)
        version_files.append(previous)
        return previous
    return None


def save_submission_version(submission, destination):
    destination = Path(destination)
    versions = destination / "versions"
    versions.mkdir(parents=True, exist_ok=True)
    current = destination / "submission.csv"
    archive_current_submission(destination)
    version_files = list(versions.glob("submission_v*.csv"))

    numbers = [int(path.stem.removeprefix("submission_v")) for path in version_files]
    version_path = versions / f"submission_v{max(numbers, default=0) + 1:03}.csv"
    submission.to_csv(version_path, index=False)
    shutil.copy2(version_path, current)
    return version_path


def write_submission(name, train_df, tuning=None):
    features = [c for c in train_df.columns if c not in {"id", "is_canceled"}]
    test_df = pd.read_csv(ROOT / "data" / "test.csv", dtype={"id": str})
    y_train = train_df["is_canceled"].to_numpy(dtype=np.int8)
    prep = SharedPreprocessor().fit(train_df[features])
    X_train = prep.transform(train_df[features])
    X_test = prep.transform(test_df[features])
    configuration = tuning or {
        "max_depth": 8, "min_samples_split": 80, "min_samples_leaf": 40,
    }
    model = BaselineTree(criterion=VARIANTS[name][0], **{
        key: configuration[key]
        for key in ("max_depth", "min_samples_split", "min_samples_leaf")
    }).fit(X_train, y_train)
    if tuning:
        if "alpha" in tuning:
            model = model.pruned(tuning["alpha"])
        probabilities = model.predict_proba(X_test)[:, 1]
        prediction = threshold_predictions(probabilities, tuning["threshold"])
    else:
        prediction = model.predict(X_test)
    template = pd.read_csv(ROOT / "data" / "sample_submission.csv", dtype={"id": str})
    submission = build_submission(template, prediction)
    destination = OUTPUT / name
    destination.mkdir(parents=True, exist_ok=True)
    save_submission_version(submission, destination)


def run_tuned_cart(train_df, train_idx, valid_idx, baseline_score):
    features = [c for c in train_df.columns if c not in {"id", "is_canceled"}]
    y = train_df["is_canceled"].to_numpy(dtype=np.int8)
    X_train_raw = train_df.iloc[train_idx][features]
    X_valid_raw = train_df.iloc[valid_idx][features]
    y_train, y_valid = y[train_idx], y[valid_idx]

    best, pruning_results = tune_cart_pruning(X_train_raw, y_train)
    X_train, X_valid = prepare_fold(X_train_raw, X_valid_raw)
    model = BaselineTree(
        criterion="gini", max_depth=None,
        min_samples_split=2, min_samples_leaf=1,
    )
    fit_start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - fit_start
    prediction_start = time.perf_counter()
    probabilities = model.predict_proba(X_valid)[:, 1]
    row_unpruned = score_row(
        "CART maximal tree (no pruning)", y_valid,
        threshold_predictions(probabilities, best["threshold"]),
        fit_seconds + time.perf_counter() - prediction_start,
        f"max_depth=None, min_samples_split=2, min_samples_leaf=1, threshold={best['threshold']:.2f}",
    )
    prune_start = time.perf_counter()
    pruned = model.pruned(best["alpha"])
    pruned_probabilities = pruned.predict_proba(X_valid)[:, 1]
    row_pruned = score_row(
        "CART maximal tree + cost-complexity pruning", y_valid,
        threshold_predictions(pruned_probabilities, best["threshold"]),
        fit_seconds + time.perf_counter() - prune_start,
        f"max_depth=None, min_samples_split=2, min_samples_leaf=1, threshold={best['threshold']:.2f}, "
        f"ccp_alpha={best['alpha']}, leaves={pruned.leaf_count()}",
    )
    rows = [row_unpruned, row_pruned]
    destination = OUTPUT / "CART"
    destination.mkdir(parents=True, exist_ok=True)
    pruning_results.to_csv(destination / "pruning.csv", index=False)
    (destination / "pruning.md").write_text(
        "# CART cost-complexity pruning\n\n"
        "Each fold grows a maximal tree (`max_depth=None`, `min_samples_split=2`, `min_samples_leaf=1`), then evaluates cost-complexity-pruned copies.\n\n"
        f"Selected `ccp_alpha={best['alpha']}` and threshold **{best['threshold']:.2f}** by 5-fold stratified CV, "
        f"with macro F1 **{best['macro_f1']:.4f}** and mean leaves **{best['mean_leaves']:.1f}**.\n\n"
        f"Holdout macro F1: **{row_unpruned['validation_macro_f1']:.4f}** before pruning; "
        f"**{row_pruned['validation_macro_f1']:.4f}** after pruning ({pruned.leaf_count()} leaves), versus original baseline **{baseline_score:.4f}**.\n\n"
        "Alpha and threshold are selected on training folds only; the shared validation split remains a final holdout. Candidates are in `pruning.csv`.\n",
        encoding="utf-8",
    )
    best.update({
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
    })
    return rows, best


def run_variant(name, train_df, train_idx, valid_idx):
    criterion, sklearn_criterion, alpha = VARIANTS[name]
    features = [c for c in train_df.columns if c not in {"id", "is_canceled"}]
    y = train_df["is_canceled"].to_numpy(dtype=np.int8)
    X_train_raw = train_df.iloc[train_idx][features]
    X_valid_raw = train_df.iloc[valid_idx][features]
    y_train, y_valid = y[train_idx], y[valid_idx]

    prep = SharedPreprocessor().fit(X_train_raw)
    X_train = prep.transform(X_train_raw)
    X_valid = prep.transform(X_valid_raw)
    configuration = "max_depth=8, min_samples_split=80, min_samples_leaf=40"

    start = time.perf_counter()
    model = BaselineTree(criterion=criterion).fit(X_train, y_train)
    prediction = model.predict(X_valid)
    rows = [score_row(f"{name} from scratch", y_valid, prediction,
                      time.perf_counter() - start, configuration)]

    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    X_train_sklearn = encoder.fit_transform(X_train[prep.categorical]) if prep.categorical else X_train[prep.numeric].to_numpy()
    X_valid_sklearn = encoder.transform(X_valid[prep.categorical]) if prep.categorical else X_valid[prep.numeric].to_numpy()
    if prep.categorical and prep.numeric:
        X_train_sklearn = np.column_stack([X_train[prep.numeric].to_numpy(), X_train_sklearn])
        X_valid_sklearn = np.column_stack([X_valid[prep.numeric].to_numpy(), X_valid_sklearn])

    sklearn_model = DecisionTreeClassifier(
        criterion=sklearn_criterion, ccp_alpha=alpha, max_depth=8,
        min_samples_split=80, min_samples_leaf=40, random_state=42,
    )
    start = time.perf_counter()
    sklearn_model.fit(X_train_sklearn, y_train)
    sklearn_prediction = sklearn_model.predict(X_valid_sklearn)
    rows.append(score_row(f"{name} scikit-learn", y_valid, sklearn_prediction,
                          time.perf_counter() - start,
                          f"criterion={sklearn_criterion}, ccp_alpha={alpha}, {configuration}, OneHotEncoder"))
    return rows


def main(only=None):
    if only is not None and only not in VARIANTS:
        raise ValueError(f"Unknown variant: {only}")
    train_df = pd.read_csv(ROOT / "data" / "train.csv", dtype={"id": str})
    train_idx, valid_idx = validation_indices(train_df["is_canceled"])
    output = OUTPUT / only if only else OUTPUT
    output.mkdir(parents=True, exist_ok=True)
    rows = []
    for name in ([only] if only else VARIANTS):
        rows.extend(run_variant(name, train_df, train_idx, valid_idx))
        tuning = None
        if name == "CART":
            baseline_score = next(row["validation_macro_f1"] for row in rows if row["model"] == "CART from scratch")
            tuned_rows, tuning = run_tuned_cart(train_df, train_idx, valid_idx, baseline_score)
            rows.extend(tuned_rows)
        write_submission(name, train_df, tuning)
    results = pd.DataFrame(rows).sort_values("validation_macro_f1", ascending=False)
    results.to_csv(output / "validation.csv", index=False)
    lines = [
        "# Perbandingan baseline Decision Tree",
        "",
        "Semua model memakai split stratified kontrak README (seed 42), preprocessing yang sama, dan macro F1 validasi. Fitur test hanya digunakan untuk membuat submission, bukan untuk seleksi model atau evaluasi. Implementasi dari nol menerima fitur kategori nominal langsung; pembanding scikit-learn memakai OneHotEncoder karena estimator itu memerlukan input numerik.",
        "",
        f"Split: {len(train_idx)} baris latih, {len(valid_idx)} baris validasi. Target validasi: 0={int((train_df.iloc[valid_idx]['is_canceled'] == 0).sum())}, 1={int((train_df.iloc[valid_idx]['is_canceled'] == 1).sum())}.",
        "",
        "| Model | Macro F1 | Precision kelas 0 | Recall kelas 0 | Precision kelas 1 | Recall kelas 1 | TN | FP | FN | TP | Detik |",
        "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |",
    ]
    for row in results.to_dict(orient="records"):
        lines.append(
            f"| {row['model']} | {row['validation_macro_f1']:.4f} | {row['class_0_precision']:.4f} | {row['class_0_recall']:.4f} | {row['class_1_precision']:.4f} | {row['class_1_recall']:.4f} | {row['tn']} | {row['fp']} | {row['fn']} | {row['tp']} | {row['seconds']:.2f} |"
        )
    lines += ["", "## Preprocessing bersama", "", "Imputasi median numerik dan penanda kategori kosong/asing dipelajari hanya dari bagian latih. `id` tidak menjadi fitur. Tiga baseline scratch memakai `max_depth=8`, `min_samples_split=80`, dan `min_samples_leaf=40`; eksperimen CART tambahan menumbuhkan pohon maksimal sebelum pruning.", "", "## Definisi baseline", "", "- ID3: information gain, kategori nominal multi-cabang, fitur numerik dengan threshold biner.", "- C4.5: gain ratio, kategori nominal multi-cabang, fitur numerik dengan threshold biner. Pembanding scikit-learn memakai entropy dan `ccp_alpha=0.001` sesuai spesifikasi.", "- CART: Gini, split kategori biner, fitur numerik dengan threshold biner. CART maksimal memakai `max_depth=None`, `min_samples_split=2`, `min_samples_leaf=1`, lalu cost-complexity pruning.", "", "Skor dan konfigurasi tiap model ada di `validation.csv`; kandidat pruning ada di `CART/pruning.csv`.", ""]
    (output / "validation.md").write_text("\n".join(lines), encoding="utf-8")
    print(results[["model", "validation_macro_f1", "seconds"]].to_string(index=False))
    print(f"\nResults: {output}")


## Jalankan end-to-end

Mengevaluasi baseline scratch dan sklearn, memilih alpha/threshold CART melalui 5-fold CV pada train partition, mengukur holdout, lalu menyimpan laporan dan submission berversi.

In [ ]:
main()

validation = pd.read_csv(OUTPUT / "validation.csv")
pruning = pd.read_csv(OUTPUT / "CART" / "pruning.csv")
display(validation[["model", "validation_macro_f1", "seconds", "configuration"]])
display(pruning.sort_values("macro_f1", ascending=False).head(10))


## Ringkasan hasil dan pemeriksaan submission

Submission terbaru tetap berada di subfolder algoritma. Setiap eksekusi menyimpan snapshot bernomor pada `versions/`. Data test hanya digunakan untuk prediksi submission, bukan seleksi atau evaluasi model.

In [ ]:
cart = validation.loc[validation["model"] == "CART maximal tree + cost-complexity pruning"].iloc[0]
plain = validation.loc[validation["model"] == "CART maximal tree (no pruning)"].iloc[0]
assert cart["validation_macro_f1"] > plain["validation_macro_f1"]
assert len(pruning) == 11 * 31
for name in VARIANTS:
    current = pd.read_csv(OUTPUT / name / "submission.csv", dtype={"id": str})
    test = pd.read_csv(ROOT / "data" / "test.csv", dtype={"id": str})
    template = pd.read_csv(ROOT / "data" / "sample_submission.csv", dtype={"id": str})
    assert len(current) == len(test)
    assert current.columns.tolist() == template.columns.tolist()
    assert current["id"].tolist() == test["id"].tolist() == template["id"].tolist()
    assert set(current["is_canceled"].unique()) <= {0, 1}
    assert list((OUTPUT / name / "versions").glob("submission_v*.csv"))
print(f"CART holdout macro F1: {plain['validation_macro_f1']:.4f} -> {cart['validation_macro_f1']:.4f}")
print("Submission versioning and row/column/ID alignment verified for ID3, C4.5, CART.")
